# Hybrid Eigen-SVD + EfficientNet-B0 + Multi-Scale Swin Attention — CIFAR-100

Pipeline: image -> **EigenTransform** (live, differentiable, per-channel top-k SVD reconstruction, replaces raw pixels) -> **EfficientNet-B0** (trained from scratch, since the eigen-transformed input has nothing to do with ImageNet pixel statistics) -> three feature maps tapped at three EfficientNet-B0 stages -> **A1 / A2 / A3**: Swin-style window attention per stage, with real shifted-window pairing (W-MSA then SW-MSA) at A1/A2, single block at A3 since it's already one window covering the whole feature map -> **Global Attention Pooling** (learned, softmax-weighted, not a plain average) per branch -> concat -> classifier.

**Note on resolution:** CIFAR-100 images are natively 32x32. The model's stage taps (28x28 / 14x14 / 7x7, all divisible by window_size=7) assume 224x224 input, so this notebook resizes CIFAR-100 up to 224x224 to keep that architecture intact. That's a real compute cost — each image is being upsampled 7x per side, and the live per-image SVD scales with resolution too. One thing that softens this: since the *source* images are only 32x32, the resized 224x224 version rarely has genuine information above roughly rank 32 anyway, so `eigen_rank=32` isn't as aggressive a truncation as it would be on a real 224x224 photo. If training turns out too slow, a native low-resolution redesign (retapping EfficientNet-B0's earlier stages, smaller window size) is a reasonable follow-up — just say the word.

In [1]:
!pip install timm -q

In [11]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import timm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


## Data — CIFAR-100

Resized to 224x224 for the reasons above. No CIFAR mean/std normalization is applied here — the model's `post_eigen_norm` (a `BatchNorm2d` right after the eigen-transform) learns its own scale for whatever the eigen-transform outputs, so pre-normalizing raw pixels first would just be undone anyway.

In [12]:
IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomCrop(IMG_SIZE, padding=16, padding_mode="reflect"),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

In [13]:
DATA_ROOT = "./data"
os.makedirs(DATA_ROOT, exist_ok=True)
train_full_aug = datasets.CIFAR100(root=DATA_ROOT, train=True, download=True, transform=train_transform)
train_full_eval = datasets.CIFAR100(root=DATA_ROOT, train=True, download=True, transform=eval_transform)
test_ds = datasets.CIFAR100(root=DATA_ROOT, train=False, download=True, transform=eval_transform)

Files already downloaded and verified
Files already downloaded and verified
Files already downloaded and verified


## Train / validation split

A held-out slice of the training set, evaluated without augmentation (`train_full_eval`'s transform) so validation accuracy isn't inflated or noised by augmentation.

In [14]:
VAL_FRACTION = 0.1
num_train_total = len(train_full_aug)
indices = list(range(num_train_total))
random.Random(SEED).shuffle(indices)
num_val = int(num_train_total * VAL_FRACTION)
val_indices = indices[:num_val]
train_indices = indices[num_val:]

train_ds = Subset(train_full_aug, train_indices)
val_ds = Subset(train_full_eval, val_indices)

print(f"train: {len(train_ds)}  val: {len(val_ds)}  test: {len(test_ds)}")

train: 45000  val: 5000  test: 10000


In [15]:
BATCH_SIZE = 32
NUM_WORKERS = 8  # tune to your CPU's core count

# persistent_workers=True avoids respawning the worker processes at the start
# of every single epoch -- on Windows (spawn-based multiprocessing) that
# respawn cost is real and adds dead time before each epoch even starts.
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                           pin_memory=True, drop_last=True, persistent_workers=(NUM_WORKERS > 0))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                         pin_memory=True, persistent_workers=(NUM_WORKERS > 0))
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                          pin_memory=True, persistent_workers=(NUM_WORKERS > 0))

## Model

Same architecture validated earlier: eigen-transform replaces the raw input, EfficientNet-B0 runs from scratch, multi-scale Swin attention with real shifted-window pairing at A1/A2, learned Global Attention Pooling per branch.

In [16]:
class EigenTransform(nn.Module):
    """Live, differentiable per-channel low-rank SVD reconstruction. Replaces
    the raw pixel input; same (B, C, H, W) shape in/out.

    Uses torch.svd_lowrank (randomized, targets rank k directly) rather than
    a full economy SVD followed by truncation. Both target the same object
    -- the best rank-k approximation -- but full-SVD-then-truncate wastefully
    computes H-k singular components it immediately discards. On structured,
    image-like data the two give reconstruction error within ~0.1% of each
    other; on GPU, svd_lowrank is also far cheaper, since it is built from
    matmuls and QR (which parallelize well) rather than the sequential
    bidiagonalization full SVD relies on.

    Forced to run in fp32 regardless of any surrounding autocast (mixed
    precision) context, since the randomized SVD is not numerically safe in
    fp16/bf16.
    """
    def __init__(self, rank: int = 32, niter: int = 4):
        super().__init__()
        self.rank = rank
        self.niter = niter

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        k = min(self.rank, H, W)
        x_flat = x.reshape(B * C, H, W)
        with torch.autocast(device_type=x.device.type, enabled=False):
            x_flat_fp32 = x_flat.float()
            U, S, V = torch.svd_lowrank(x_flat_fp32, q=k, niter=self.niter)
            recon = torch.einsum("bhk,bk,bkw->bhw", U, S, V.transpose(-2, -1))
        return recon.reshape(B, C, H, W).to(x.dtype)


def window_partition(x: torch.Tensor, window_size: int) -> torch.Tensor:
    B, H, W, C = x.shape
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous()
    return windows.view(-1, window_size, window_size, C)


def window_reverse(windows: torch.Tensor, window_size: int, H: int, W: int) -> torch.Tensor:
    B = windows.shape[0] // (H // window_size * W // window_size)
    x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous()
    return x.view(B, H, W, -1)


def compute_shift_mask(H: int, W: int, window_size: int, shift_size: int, device) -> torch.Tensor:
    """Standard Swin attention mask so cyclic-shifted windows don't let
    tokens attend across the artificial wrap-around boundary."""
    img_mask = torch.zeros((1, H, W, 1), device=device)
    h_slices = (slice(0, -window_size), slice(-window_size, -shift_size), slice(-shift_size, None))
    w_slices = (slice(0, -window_size), slice(-window_size, -shift_size), slice(-shift_size, None))
    cnt = 0
    for h in h_slices:
        for w in w_slices:
            img_mask[:, h, w, :] = cnt
            cnt += 1
    mask_windows = window_partition(img_mask, window_size).view(-1, window_size * window_size)
    attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
    attn_mask = attn_mask.masked_fill(attn_mask != 0, float(-100.0)).masked_fill(attn_mask == 0, float(0.0))
    return attn_mask


class WindowAttention(nn.Module):
    def __init__(self, dim: int, window_size: int, num_heads: int):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=True)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        if mask is not None:
            nW = mask.shape[0]
            attn = attn.view(B_ // nW, nW, self.num_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(B_, self.num_heads, N, N)
        attn = attn.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        return self.proj(out)


class SwinStyleBlock(nn.Module):
    def __init__(self, dim: int, window_size: int = 7, num_heads: int = 4,
                 shift_size: int = 0, mlp_ratio: float = 4.0):
        super().__init__()
        self.window_size = window_size
        self.shift_size = shift_size
        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention(dim, window_size, num_heads)
        self.norm2 = nn.LayerNorm(dim)
        hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim))
        self._mask_cache = {}

    def _get_mask(self, H: int, W: int, device) -> torch.Tensor:
        if self.shift_size == 0:
            return None
        key = (H, W, str(device))
        if key not in self._mask_cache:
            self._mask_cache[key] = compute_shift_mask(H, W, self.window_size, self.shift_size, device)
        return self._mask_cache[key]

    def forward(self, x: torch.Tensor, H: int, W: int) -> torch.Tensor:
        B, N, C = x.shape
        assert H % self.window_size == 0 and W % self.window_size == 0
        shortcut = x
        x = self.norm1(x).view(B, H, W, C)
        if self.shift_size > 0:
            x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
        windows = window_partition(x, self.window_size).view(-1, self.window_size * self.window_size, C)
        mask = self._get_mask(H, W, x.device)
        attn_windows = self.attn(windows, mask=mask)
        attn_windows = attn_windows.view(-1, self.window_size, self.window_size, C)
        x = window_reverse(attn_windows, self.window_size, H, W)
        if self.shift_size > 0:
            x = torch.roll(x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))
        x = x.view(B, N, C)
        x = shortcut + x
        return x + self.mlp(self.norm2(x))


class AttentionPool(nn.Module):
    """Global Attention Pooling: learned, softmax-weighted sum over tokens."""
    def __init__(self, dim: int):
        super().__init__()
        self.score = nn.Linear(dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        weights = torch.softmax(self.score(x), dim=1)
        return (weights * x).sum(dim=1)


class HybridEigenSwinNet(nn.Module):
    def __init__(self, num_classes: int = 100, eigen_rank: int = 32, embed_dim: int = 192,
                 window_size: int = 7, num_heads: int = 4, img_size: int = 224):
        super().__init__()
        self.eigen = EigenTransform(rank=eigen_rank)
        self.post_eigen_norm = nn.BatchNorm2d(3)

        self.backbone = timm.create_model(
            "efficientnet_b0", pretrained=False, features_only=True, out_indices=(2, 3, 4)
        )
        stage_channels = self.backbone.feature_info.channels()
        stage_reductions = self.backbone.feature_info.reduction()
        stage_hw = [img_size // r for r in stage_reductions]

        self.proj = nn.ModuleList([nn.Conv2d(c, embed_dim, kernel_size=1) for c in stage_channels])

        self.swin_blocks = nn.ModuleList()
        self.branch_uses_shift = []
        for hw in stage_hw:
            assert hw % window_size == 0, f"stage size {hw} not divisible by window_size {window_size}"
            num_windows = (hw // window_size) * (hw // window_size)
            if num_windows > 1:
                blocks = nn.ModuleList([
                    SwinStyleBlock(embed_dim, window_size, num_heads, shift_size=0),
                    SwinStyleBlock(embed_dim, window_size, num_heads, shift_size=window_size // 2),
                ])
                self.branch_uses_shift.append(True)
            else:
                blocks = nn.ModuleList([SwinStyleBlock(embed_dim, window_size, num_heads, shift_size=0)])
                self.branch_uses_shift.append(False)
            self.swin_blocks.append(blocks)

        self.pools = nn.ModuleList([AttentionPool(embed_dim) for _ in stage_channels])
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim * len(stage_channels)),
            nn.Linear(embed_dim * len(stage_channels), num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.eigen(x)
        x = self.post_eigen_norm(x)
        feats = self.backbone(x)
        branch_outs = []
        for feat, proj, blocks, pool in zip(feats, self.proj, self.swin_blocks, self.pools):
            f = proj(feat)
            B, C, H, W = f.shape
            tokens = f.flatten(2).transpose(1, 2)
            for block in blocks:
                tokens = block(tokens, H, W)
            branch_outs.append(pool(tokens))
        fused = torch.cat(branch_outs, dim=1)
        return self.classifier(fused)

In [17]:
NUM_CLASSES = 100  # CIFAR-100

model = HybridEigenSwinNet(
    num_classes=NUM_CLASSES,
    eigen_rank=32,
    embed_dim=192,
    window_size=7,
    num_heads=4,
    img_size=IMG_SIZE,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Total trainable parameters: {n_params:,}")

# sanity check on one real batch before committing to a full training run
xb, yb = next(iter(train_loader))
xb, yb = xb.to(device), yb.to(device)
out = model(xb)
print("output shape:", out.shape)

Total trainable parameters: 5,970,345
output shape: torch.Size([32, 100])


## Optimizer, scheduler, and a quick benchmark before committing to a full run

Set up here (rather than right before the training loop) so the benchmark cell below can use the real `optimizer`/`scaler`/`criterion` instead of standing up throwaway copies.

In [18]:
EPOCHS = 15
LR = 3e-4
WEIGHT_DECAY = 0.05
USE_AMP = (device.type == "cuda")

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = torch.amp.GradScaler(device.type, enabled=USE_AMP)

# Mixed precision speeds up the conv/attention/matmul-heavy parts of the model
# on Tensor Cores (roughly halves their cost) and is standard, accuracy-neutral
# practice. EigenTransform ignores this and always runs its SVD in fp32
# internally regardless of this autocast context, so numerical stability of
# the eigen-transform itself is unaffected either way.

In [19]:
import time

model.train()
data_iter = iter(train_loader)

for _ in range(3):  # warmup: skips one-time CUDA kernel compilation / worker startup cost
    xb, yb = next(data_iter)
    xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
    optimizer.zero_grad()
    with torch.amp.autocast(device.type, enabled=USE_AMP):
        loss = criterion(model(xb), yb)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

if device.type == "cuda":
    torch.cuda.synchronize()
n_steps = 20
t0 = time.time()
for _ in range(n_steps):
    xb, yb = next(data_iter)
    xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
    optimizer.zero_grad()
    with torch.amp.autocast(device.type, enabled=USE_AMP):
        loss = criterion(model(xb), yb)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
if device.type == "cuda":
    torch.cuda.synchronize()

per_step = (time.time() - t0) / n_steps
print(f"{per_step*1000:.0f} ms/step -> ~{per_step*len(train_loader)/60:.1f} min for one training epoch")

299 ms/step -> ~7.0 min for one training epoch


## Training loop

In [20]:
def run_epoch(loader, train: bool):
    model.train(mode=train)
    total_loss, total_correct, total_count = 0.0, 0, 0
    torch.set_grad_enabled(train)
    for xb, yb in loader:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        if train:
            optimizer.zero_grad()
        try:
            with torch.amp.autocast(device.type, enabled=USE_AMP):
                out = model(xb)
                loss = criterion(out, yb)
            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            print(f"OOM at batch size {xb.size(0)} -- lower BATCH_SIZE and rerun.")
            raise
        total_loss += loss.item() * xb.size(0)
        total_correct += (out.argmax(dim=1) == yb).sum().item()
        total_count += xb.size(0)
    torch.set_grad_enabled(True)
    return total_loss / total_count, total_correct / total_count


best_val_acc = 0.0
for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    scheduler.step()
    print(f"epoch {epoch:02d}/{EPOCHS}  train_loss {train_loss:.4f}  train_acc {train_acc:.4f}  "
          f"val_loss {val_loss:.4f}  val_acc {val_acc:.4f}")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_hybrid_eigen_model.pt")

print("best val acc:", best_val_acc)

epoch 01/15  train_loss 3.8207  train_acc 0.1439  val_loss 3.3446  val_acc 0.2476
epoch 02/15  train_loss 3.0786  train_acc 0.3112  val_loss 2.8160  val_acc 0.3800
epoch 03/15  train_loss 2.6736  train_acc 0.4230  val_loss 2.5196  val_acc 0.4676
epoch 04/15  train_loss 2.4049  train_acc 0.4934  val_loss 2.3694  val_acc 0.5108
epoch 05/15  train_loss 2.2057  train_acc 0.5552  val_loss 2.2332  val_acc 0.5440
epoch 06/15  train_loss 2.0436  train_acc 0.6053  val_loss 2.1366  val_acc 0.5734
epoch 07/15  train_loss 1.8976  train_acc 0.6501  val_loss 2.0760  val_acc 0.5986
epoch 08/15  train_loss 1.7697  train_acc 0.6946  val_loss 2.0104  val_acc 0.6200
epoch 09/15  train_loss 1.6544  train_acc 0.7329  val_loss 1.9706  val_acc 0.6374
epoch 10/15  train_loss 1.5513  train_acc 0.7712  val_loss 1.9278  val_acc 0.6372
epoch 11/15  train_loss 1.4619  train_acc 0.8046  val_loss 1.9144  val_acc 0.6500
epoch 12/15  train_loss 1.3872  train_acc 0.8332  val_loss 1.8918  val_acc 0.6650
epoch 13/15  tra

## Test evaluation

In [21]:
model.load_state_dict(torch.load("best_hybrid_eigen_model.pt", map_location=device))
test_loss, test_acc = run_epoch(test_loader, train=False)
print(f"test_loss {test_loss:.4f}  test_acc {test_acc:.4f}")

C:\Users\adnan\AppData\Local\Temp\ipykernel_8360\3971303063.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_hybrid_eigen_model.pt"

test_loss 1.8488  test_acc 0.6750


In [22]:
import torch

device = next(model.parameters()).device
x = torch.randn(4, 3, 224, 224, device=device)

with torch.no_grad():
    out = model.eigen(x)

print(f"input shape:  {tuple(x.shape)}")
print(f"output shape: {tuple(out.shape)}")
print(f"shapes match: {out.shape == x.shape}")
print(f"dtype match:  {out.dtype == x.dtype}  (in={x.dtype}, out={out.dtype})")
print(f"has NaN:      {torch.isnan(out).any().item()}")
print(f"has Inf:      {torch.isinf(out).any().item()}")

assert out.shape == x.shape, f"shape mismatch! expected {x.shape}, got {out.shape}"
print("\nEigenTransform shape check: PASSED")

# bonus: confirm it's actually doing a low-rank reconstruction, not just
# preserving shape trivially
svals = torch.linalg.svdvals(out[0, 0])  # one image, one channel
effective_rank = (svals > svals.max() * 1e-4).sum().item()
print(f"effective rank of that channel: {effective_rank}  (eigen_rank={model.eigen.rank})")

input shape:  (4, 3, 224, 224)
output shape: (4, 3, 224, 224)
shapes match: True
dtype match:  True  (in=torch.float32, out=torch.float32)
has NaN:      False
has Inf:      False

EigenTransform shape check: PASSED
effective rank of that channel: 32  (eigen_rank=32)


In [24]:
import torch

device = next(model.parameters()).device
torch.set_printoptions(precision=3, sci_mode=False, linewidth=120)

xb, yb = next(iter(val_loader))
xb = xb[:1].to(device)  # one image, keeps the printout readable

with torch.no_grad():
    x_eigen = model.eigen(xb)                    # after EigenTransform
    x_cnn_input = model.post_eigen_norm(x_eigen)  # after BatchNorm -- what EfficientNet-B0 actually sees

img, ch, row, col = 0, 0, 100, 100  # first image, red channel, a 6x6 patch away from the edges
sl = slice(row, row + 6)

print(f"tensor shape at each stage: {tuple(xb.shape)}\n")

print("1) RAW pixels (before any transform):")
print(xb[img, ch, sl, sl])

print("\n2) AFTER eigen transform (rank-{} SVD reconstruction):".format(model.eigen.rank))
print(x_eigen[img, ch, sl, sl])

print("\n3) AFTER post_eigen_norm (actual CNN input):")
print(x_cnn_input[img, ch, sl, sl])

print("\ndifference, raw vs eigen (same patch):")
print(xb[img, ch, sl, sl] - x_eigen[img, ch, sl, sl])

for name, t in [("raw", xb[img, ch]), ("eigen", x_eigen[img, ch]), ("cnn_input", x_cnn_input[img, ch])]:
    print(f"\n{name:10s} -> min {t.min():.4f}  max {t.max():.4f}  mean {t.mean():.4f}  std {t.std():.4f}")

tensor shape at each stage: (1, 3, 224, 224)

1) RAW pixels (before any transform):
tensor([[1.000, 1.000, 0.996, 0.992, 0.988, 0.988],
        [1.000, 1.000, 0.996, 0.992, 0.988, 0.988],
        [1.000, 1.000, 0.992, 0.988, 0.984, 0.980],
        [0.996, 0.996, 0.992, 0.984, 0.980, 0.976],
        [0.996, 0.996, 0.988, 0.980, 0.976, 0.969],
        [0.996, 0.996, 0.988, 0.980, 0.969, 0.961]], device='cuda:0')

2) AFTER eigen transform (rank-32 SVD reconstruction):
tensor([[1.000, 0.999, 0.996, 0.993, 0.990, 0.990],
        [1.000, 1.000, 0.996, 0.992, 0.989, 0.988],
        [1.000, 0.999, 0.994, 0.989, 0.984, 0.981],
        [0.997, 0.997, 0.991, 0.984, 0.979, 0.975],
        [0.996, 0.996, 0.989, 0.981, 0.974, 0.968],
        [0.996, 0.996, 0.987, 0.980, 0.971, 0.961]], device='cuda:0')

3) AFTER post_eigen_norm (actual CNN input):
tensor([[1.518, 1.516, 1.505, 1.496, 1.488, 1.486],
        [1.519, 1.519, 1.506, 1.493, 1.484, 1.480],
        [1.518, 1.517, 1.500, 1.483, 1.469, 1.457]